In [25]:
import numpy as np
#假设条件：binomial tree
u=1.1
d=0.9
S0=100
n_steps=5
k=100
r=0.03
dt=1/n_steps
# q 为Q_Quant上的概率
q=(np.exp(r*dt)-d)/(u-d)
#股价树
tree=np.zeros((n_steps+1,n_steps+1))
tree[0,0]=S0
for i in range (1,n_steps+1):
    for j in range (i):
        tree[i,j]=tree[i-1,j]*d
    tree[i,i]=tree[i-1,i-1]*u
print ("Stock Price Matrix")
print (tree)
#期权价格树
option=np.zeros((n_steps+1,n_steps+1))
#先假设都是欧洲期权
for i in range(n_steps+1):
    option[n_steps,i]=max(tree[n_steps,i]-k,0)
# 从倒数第二行开始倒着赋值
for i in range(n_steps-1,-1,-1):
    for j in range(i+1):
        #加入max(期权价格-k,hold)来变为美国期权
        Hold=q*option[i+1,j]*np.exp(-r*dt)+(1-q)*option[i+1,j+1]*np.exp(-r*dt)
        Call=max(tree[i,j]-k,0)
        option[i,j]=max(Call,Hold)
print ("Option Price Matrix")
print (option)
#delta树
delta=np.zeros((n_steps,n_steps))
for i in range(n_steps):
    for j in range(i+1):
        delta[i,j]=(option[i+1,j]-option[i+1,j+1])/(tree[i+1,j]-tree[i+1,j+1])
print ("delta Tree")
print (delta)
#gamma树
gamma=np.zeros((n_steps-1,n_steps-1))
for i in range(n_steps-1):
    for j in range(i+1):
        gamma[i,j]=(delta[i+1,j]-delta[i+1,j+1])/(tree[i+1,j]-tree[i+1,j+1])
print ("gamma Tree")
print (gamma)

Stock Price Matrix
[[100.      0.      0.      0.      0.      0.   ]
 [ 90.    110.      0.      0.      0.      0.   ]
 [ 81.     99.    121.      0.      0.      0.   ]
 [ 72.9    89.1   108.9   133.1     0.      0.   ]
 [ 65.61   80.19   98.01  119.79  146.41    0.   ]
 [ 59.049  72.171  88.209 107.811 131.769 161.051]]
Option Price Matrix
[[ 7.88752123  0.          0.          0.          0.          0.        ]
 [ 3.27518325 13.19156544  0.          0.          0.          0.        ]
 [ 0.79603635  6.11377384 21.34473649  0.          0.          0.        ]
 [ 0.          1.70421406 11.16635447 33.1         0.          0.        ]
 [ 0.          0.          3.64850874 19.79       46.41        0.        ]
 [ 0.          0.          0.          7.811      31.769      61.051     ]]
delta Tree
[[ 0.49581911  0.          0.          0.          0.        ]
 [ 0.29542986  0.69231648  0.          0.          0.        ]
 [ 0.1051984   0.47788588  0.90634899  0.          0.        ]
 [-